# tosave · Colab ile Android APK derle

Hücreleri sırayla çalıştır. GPU gerekmez; CPU çalışma zamanı yeterlidir.

Bu defter public `trdevLabs/SaveWhy` reposundan kodu alır, JDK 17 ve sabitlenmiş Android araçlarını kurar, birim testlerini ve lint kontrolünü çalıştırır, ARM64 debug APK oluşturur. GitHub token gerekmez. SDK kurulum hücresi [Android SDK lisanslarını](https://developer.android.com/studio/terms) kabul eder; çalıştırmadan önce şartları incele.

Üretilen APK geliştirme içindir; Play Store için imzalı release/AAB değildir. Colab çalışma zamanı silinmeden APK'yı indir. Colab debug anahtarı yenilenirse önceki APK üzerine güncelleme kurulamayabilir; uygulamayı silmeden önce tosave menüsünden JSON yedeği al.


## 1. JDK kurulumu ve repoyu al

In [ ]:
import os, pathlib, subprocess

def run(*args, **kwargs):
    subprocess.run(args, check=True, **kwargs)

run('apt-get', 'update', '-qq')
run('apt-get', 'install', '-y', '-qq', 'openjdk-17-jdk-headless', 'curl', 'unzip', 'git')
candidates = sorted(pathlib.Path('/usr/lib/jvm').glob('java-17-openjdk-*/bin/javac'))
if not candidates:
    raise RuntimeError('JDK 17 kurulumu bulunamadı.')
javac = candidates[0].resolve()
os.environ['JAVA_HOME'] = str(javac.parent.parent)
os.environ['PATH'] = str(javac.parent) + ':' + os.environ['PATH']
repo = pathlib.Path('/content/SaveWhy')
if not repo.exists():
    run('git', 'clone', 'https://github.com/trdevLabs/SaveWhy.git', str(repo))
else:
    if subprocess.check_output(['git', '-C', str(repo), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('Repo içinde değişiklik var. Değişikliklerini kaydet veya yeni çalışma zamanı aç.')
    run('git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main')
run('git', '-C', str(repo), 'rev-parse', 'HEAD')


## 2. Test et ve APK derle

İlk çalıştırmada SDK ve Gradle indirilir; birkaç dakika sürebilir. Hata olursa sonraki indirme hücresini çalıştırma, bu hücrenin hata çıktısını kontrol et.

In [ ]:
os.environ['SAVEWHY_BUILD_CACHE'] = '/content/savewhy-tools'
os.environ['ACCEPT_ANDROID_SDK_LICENSES'] = 'yes'
run('bash', str(repo / 'scripts/build_colab.sh'))


## 3. APK indir

Bu hücre APK özetini son başarılı derlemeyle karşılaştırır. Telefona kurduktan sonra Paylaş → tosave akışını dene.

In [ ]:
import hashlib, json
from google.colab import files
marker = repo / '.savewhy-build-success'
if not marker.exists():
    raise RuntimeError('Başarılı derleme yok. Önce 2. hücreyi tamamla.')
info = json.loads(marker.read_text())
current = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
if current != info['commit']:
    raise RuntimeError('Kaynak commit değişmiş. Önce tekrar derle.')
apk = repo / 'app/build/outputs/apk/debug/app-debug.apk'
if hashlib.sha256(apk.read_bytes()).hexdigest() != info['sha256']:
    raise RuntimeError('APK değişmiş. Tekrar derle.')
print('Derlenen commit:', info['commit'])
files.download(str(apk))
